In [ ]:
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Load tất cả kết quả ───────────────────────────────────────────────────────
result_files = glob.glob("../data/results/*.json")
all_records  = []

for f in result_files:
    with open(f) as fp:
        data = json.load(fp)
    all_records.extend(data["results"])

df = pd.DataFrame(all_records)
print(f"Total records: {len(df)}")
print(df[["model", "attack_type", "verdict", "is_success", "eval_method"]].head(10))

# ── Bảng ASR theo model × attack_type ────────────────────────────────────────
asr_table = (
    df.groupby(["model", "attack_type"])["is_success"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "success", "count": "total"})
)
asr_table["ASR (%)"] = (asr_table["success"] / asr_table["total"] * 100).round(2)
print("\n=== ASR Table ===")
print(asr_table.to_string())

# ── Heatmap ASR + Score distribution ─────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

pivot = asr_table["ASR (%)"].unstack(level="attack_type").fillna(0)
sns.heatmap(pivot, annot=True, fmt=".1f", cmap="RdYlGn_r",
            linewidths=0.5, ax=axes[0],
            cbar_kws={"label": "ASR (%)"})
axes[0].set_title("ASR (%) — Model × Attack Type")

for model in df["model"].unique():
    df[df["model"] == model]["score"].hist(
        alpha=0.6, bins=5, label=model, ax=axes[1]
    )
axes[1].set_xlabel("Judge Score (1=Refused → 5=Full Bypass)")
axes[1].set_ylabel("Count")
axes[1].set_title("Score Distribution by Model")
axes[1].legend()

plt.tight_layout()
plt.savefig("asr_analysis.png", dpi=150)
plt.show()

# ── Eval method breakdown ─────────────────────────────────────────────────────
print("\n=== Evaluation Method Used ===")
print(df.groupby(["model", "eval_method"])["is_success"].mean().round(3))

# ── PAIR: phân tích vòng lặp ──────────────────────────────────────────────────
pair_df = df[df["attack_type"] == "pair"].dropna(subset=["pair_iterations"])
if not pair_df.empty:
    print("\n=== PAIR Iterations Statistics ===")
    print(pair_df.groupby("model")["pair_iterations"].describe().round(2))
    print(f"\nPAIR overall success rate: {pair_df['pair_success'].mean():.1%}")